In [1]:
import os

os.environ["CC"] = "/usr/bin/gcc"
os.environ["CXX"] = "/usr/bin/g++"

import cmdstanpy
cmdstanpy.install_cmdstan(verbose=True)

/u/zwu1/.conda/envs/ou/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


CmdStan install directory: /u/zwu1/.cmdstan
CmdStan version 2.38.0 already installed
Test model compilation

--- Translating Stan model to C++ code ---
bin/stanc  --o=examples/bernoulli/bernoulli.hpp examples/bernoulli/bernoulli.stan

--- Compiling C++ code ---
/usr/bin/g++ -std=c++17 -pthread -D_REENTRANT -Wno-sign-compare -Wno-ignored-attributes -Wno-class-memaccess      -I stan/lib/stan_math/lib/tbb_2020.3/include    -O3 -I src -I stan/src -I stan/lib/rapidjson_1.1.0/ -I lib/CLI11-1.9.1/ -I stan/lib/stan_math/ -I stan/lib/stan_math/lib/eigen_3.4.0 -I stan/lib/stan_math/lib/boost_1.87.0 -I stan/lib/stan_math/lib/sundials_6.1.1/include -I stan/lib/stan_math/lib/sundials_6.1.1/src/sundials    -DBOOST_DISABLE_ASSERTS          -c -Wno-ignored-attributes   -x c++ -o examples/bernoulli/bernoulli.o examples/bernoulli/bernoulli.hpp

--- Linking model ---
/usr/bin/g++ -std=c++17 -pthread -D_REENTRANT -Wno-sign-compare -Wno-ignored-attributes -Wno-class-memaccess      -I stan/lib/stan_math/lib

True

In [4]:
import numpy as np
import os
from cmdstanpy import CmdStanModel


stan_file = "ultra_model_fix_mean.stan"

print("Compiling Stan Model (this may take a minute)...")
model = CmdStanModel(stan_file=stan_file)


# ==========================================
# 2. Simulate Synthetic Data
# ==========================================
print("Generating Synthetic Data...")
np.random.seed(42)

Nsub = 15          # Number of individuals
obs_per_sub = 10   # Observations per individual
N = Nsub * obs_per_sub
K = 7              # 3 binary + 4 ordinal items
R = 2              # Latent factors
p = 2              # Covariates
ncate = 4          # Number of categories for ordinal items (1 to 4)

# Subject IDs and indexing
ID = np.repeat(np.arange(1, Nsub + 1), obs_per_sub)
repme = np.full(Nsub, obs_per_sub)
cumu = np.cumsum(repme)

# Irregular time gaps (deltat = 0 for the first observation of each person)
deltat = np.random.uniform(0.5, 2.0, size=N)
deltat[::obs_per_sub] = 0.0 

# Covariates
X = np.random.normal(0, 1, size=(N, p))

# Mock categorical responses (Noise data just to test sampler mechanics)
Y = np.zeros((N, K), dtype=int)
Y[:, 0:3] = np.random.binomial(1, 0.5, size=(N, 3))         # Binary: 0 or 1
Y[:, 3:7] = np.random.randint(1, ncate + 1, size=(N, 4))    # Ordinal: 1, 2, 3, or 4

# No missing data for this test
missing_ID = np.zeros((N, K), dtype=int)

stan_data = {
    'N': N,
    'Nsub': Nsub,
    'K': K,
    'R': R,
    'p': p,
    'ID': ID.tolist(),
    'cumu': cumu.tolist(),
    'repme': repme.tolist(),
    'Y': Y.tolist(),
    'missing_ID': missing_ID.tolist(),
    'deltat': deltat.tolist(),
    'X': X.tolist(),
    'ncate4': ncate,
    'ncate5': ncate,
    'ncate6': ncate,
    'ncate7': ncate
}


# ==========================================
# 3. Fit the Model
# ==========================================
print("Starting HMC/NUTS Sampler...")
# Keeping iter_warmup and iter_sampling low for a quick local test
fit = model.sample(
    data=stan_data,
    chains=2,
    iter_warmup=200,
    iter_sampling=200,
    max_treedepth=10,
    adapt_delta=0.85,
    show_progress=True
)

# ==========================================
# 4. Output Diagnostics
# ==========================================
print("\n=== Sampling Complete ===")
summary_df = fit.summary()
print("\nSummary of structural parameters (Gamma matrix components):")
print(summary_df.loc[['L_S[1,1]', 'a_low[1]'], ['Mean', 'MCSE', 'StdDev', 'ESS_bulk', 'ESS_tail', 'R_hat']])

print("\nModel diagnostics:")
# Total number of divergent transitions across all chains
divergences = fit.divergences.sum()
print(f"\nTotal Divergent Transitions: {divergences}")

# Total times the sampler hit the max treedepth
treedepths = (fit.method_variables()['treedepth__'] >= 10).sum()
print(f"Total Max Treedepth Hits: {treedepths}")

18:11:50 - cmdstanpy - INFO - compiling stan file /nfs/nfs9/home/nobackup/zwu1/gnpc_explor/src/ultra_model_fix_mean.stan to exe file /nfs/nfs9/home/nobackup/zwu1/gnpc_explor/src/ultra_model_fix_mean


Compiling Stan Model (this may take a minute)...


18:12:59 - cmdstanpy - INFO - compiled model executable: /nfs/nfs9/home/nobackup/zwu1/gnpc_explor/src/ultra_model_fix_mean
18:12:59 - cmdstanpy - WARNING - Stan compiler has produced 1 warnings:
18:12:59 - cmdstanpy - WARNING - 
--- Translating Stan model to C++ code ---
bin/stanc --filename-in-msg=ultra_model_fix_mean.stan --o=/nfs/nfs9/home/nobackup/zwu1/gnpc_explor/src/ultra_model_fix_mean.hpp /nfs/nfs9/home/nobackup/zwu1/gnpc_explor/src/ultra_model_fix_mean.stan
Warning in 'ultra_model_fix_mean.stan', line 91, column 11 to column 22:
    Found int division:
        R * (R - 1) / 2
    Values will be rounded towards zero. If rounding is not desired you can
    write the division as
        R * (R - 1) / 2.0
    If rounding is intended please use the integer division operator %/%.

--- Compiling C++ code ---
/usr/bin/g++ -std=c++17 -pthread -D_REENTRANT -Wno-sign-compare -Wno-ignored-attributes -Wno-class-memaccess      -I stan/lib/stan_math/lib/tbb_2020.3/include    -O3 -I src -I st

Generating Synthetic Data...
Starting HMC/NUTS Sampler...


18:13:00 - cmdstanpy - INFO - CmdStan start processing
chain 2: 100%|██████████| 400/400 [00:23<00:00, 17.13it/s, (Sampling completed)]


18:13:23 - cmdstanpy - INFO - CmdStan done processing.
18:13:23 - cmdstanpy - WARNING - Non-fatal error during sampling:
Exception: ordered_logistic: Cut-points is not a valid ordered vector. The element at 2 is -289.313, but should be greater than the previous element, -289.313 (in 'ultra_model_fix_mean.stan', line 212, column 35 to column 126)
	Exception: ordered_logistic: Cut-points is not a valid ordered vector. The element at 2 is -286.243, but should be greater than the previous element, -286.243 (in 'ultra_model_fix_mean.stan', line 212, column 35 to column 126)
	Exception: ordered_logistic: Cut-points is not a valid ordered vector. The element at 2 is -71.164, but should be greater than the previous element, -71.164 (in 'ultra_model_fix_mean.stan', line 212, column 35 to column 126)
	Exception: ordered_logistic: Cut-points is not a valid ordered vector. The element at 2 is -17.0047, but should be greater than the previous element, -17.0047 (in 'ultra_model_fix_mean.stan', line



=== Sampling Complete ===

Summary of structural parameters (Gamma matrix components):
              Mean      MCSE    StdDev  ESS_bulk  ESS_tail     R_hat
L_S[1,1]  1.543750  0.041636  0.655672   254.885   317.806  0.998279
a_low[1]  0.053664  0.031480  0.545314   313.990   254.747  1.000120

Model diagnostics:

Total Divergent Transitions: 0
Total Max Treedepth Hits: 0


In [6]:
import numpy as np
import scipy.linalg as la
import pandas as pd
from cmdstanpy import CmdStanModel

# ==========================================
# 1. Define True Structural Parameters (DGP)
# ==========================================
print("Generating True Continuous-Time LOU Data...")
np.random.seed(101)

Nsub = 50          # Number of individuals
obs_per_sub = 20   # Observations per individual
N = Nsub * obs_per_sub
K = 7              # 3 binary + 4 ordinal items
R = 2              # Latent factors
p = 2              # Number of covariates
ncate = 4          # Number of categories for ordinal items (1 to 4)

# True Base Matrices for Gamma = S + A
# S must be Symmetric Positive Definite
S_true = np.array([[1.5, 0.2], 
                   [0.2, 1.0]])

# A must be Skew-Symmetric (determines oscillation)
A_true = np.array([[ 0.0,  0.5], 
                   [-0.5,  0.0]])

Gamma_true = S_true + A_true

# True Diffusion Matrix (Sigma)
Sigma_true = np.array([[1.0, 0.3], 
                       [0.3, 1.0]])

# Solve Lyapunov: Gamma * Omega + Omega * Gamma^T = Sigma
Omega_true = la.solve_continuous_lyapunov(Gamma_true, Sigma_true)

# ==========================================
# 2. Simulate the Latent OU Trajectories (xi)
# ==========================================
ID = np.repeat(np.arange(1, Nsub + 1), obs_per_sub)
repme = np.full(Nsub, obs_per_sub)
cumu = np.cumsum(repme)

# Irregular time gaps (deltat = 0 for the first observation of each person)
deltat = np.random.uniform(0.5, 3.0, size=N)
deltat[::obs_per_sub] = 0.0 

xi_true = np.zeros((R, N))

for i in range(Nsub):
    start_idx = cumu[i] - repme[i]
    
    # Time 1: Draw from stationary distribution N(0, Omega)
    xi_true[:, start_idx] = np.random.multivariate_normal(np.zeros(R), Omega_true)
    
    # Time 2 to end: Conditional autoregressive draws
    for j in range(1, repme[i]):
        idx = start_idx + j
        dt = deltat[idx]
        
        # Transition matrix: Phi = exp(-Gamma * dt)
        Phi = la.expm(-Gamma_true * dt)
        
        # Conditional Covariance: Q = Omega - Phi * Omega * Phi^T
        Q_cond = Omega_true - Phi @ Omega_true @ Phi.T
        
        # Ensure symmetric positive definite for numpy generator due to float rounding
        Q_cond = 0.5 * (Q_cond + Q_cond.T) 
        
        mu_cond = Phi @ xi_true[:, idx - 1]
        xi_true[:, idx] = np.random.multivariate_normal(mu_cond, Q_cond)

# ==========================================
# 3. Define True IRT Measurement Parameters
# ==========================================
# Intercepts for Binary Items (1-3)
theta_bin_true = np.array([-0.5, 0.2, 0.8])

# Cut-points for Ordinal Items (4-7) (Strictly increasing)
theta_ord_true = np.array([-1.5, 0.0, 1.5]) 

# Factor Loadings (lambda) - Items 1 and 4 anchored at 1.0
lambda_true = np.array([1.0, 0.9, 1.1, 1.0, 0.8, 1.0, 1.4])

# Covariate effects (beta)
X = np.random.normal(0, 1, size=(N, p))
beta_true = np.random.normal(0, 0.5, size=(K, p))

# Subject-Item Random Effects (b)
sigma_bk_true = np.random.uniform(0.2, 0.5, size=K)
b_true = np.zeros((Nsub, K))
for k in range(K):
    b_true[:, k] = np.random.normal(0, sigma_bk_true[k], size=Nsub)

# ==========================================
# 4. Generate Categorical & Ordinal Responses
# ==========================================
Y = np.zeros((N, K), dtype=int)

for i in range(N):
    sub_idx = ID[i] - 1
    
    # --- BINARY ITEMS (1-3) mapping to Factor 1 ---
    for k in range(3):
        linear_predictor = theta_bin_true[k] + np.dot(X[i], beta_true[k]) + \
                           lambda_true[k] * xi_true[0, i] + b_true[sub_idx, k]
        
        prob = 1.0 / (1.0 + np.exp(-linear_predictor)) # Inverse logit
        Y[i, k] = np.random.binomial(1, prob)
        
    # --- ORDINAL ITEMS (4-7) mapping to Factor 2 ---
    for k in range(3, 7):
        latent_utility = np.dot(X[i], beta_true[k]) + \
                         lambda_true[k] * xi_true[1, i] + b_true[sub_idx, k]
        
        # Ordered logistic uses standard logistic error
        epsilon = np.random.logistic(0, 1)
        Z = latent_utility + epsilon
        
        # Classify based on cutpoints (Produces 1, 2, 3, or 4)
        if Z <= theta_ord_true[0]:
            Y[i, k] = 1
        elif Z <= theta_ord_true[1]:
            Y[i, k] = 2
        elif Z <= theta_ord_true[2]:
            Y[i, k] = 3
        else:
            Y[i, k] = 4

missing_ID = np.zeros((N, K), dtype=int)

# ==========================================
# 5. Pack Data and Fit the Stan Model
# ==========================================
stan_data = {
    'N': N,
    'Nsub': Nsub,
    'K': K,
    'R': R,
    'p': p,
    'ID': ID.tolist(),
    'cumu': cumu.tolist(),
    'repme': repme.tolist(),
    'Y': Y.tolist(),
    'missing_ID': missing_ID.tolist(),
    'deltat': deltat.tolist(),
    'X': X.tolist(),
    'ncate4': ncate,
    'ncate5': ncate,
    'ncate6': ncate,
    'ncate7': ncate
}

print("Compiling ultra_model_fix_mean.stan...")
model = CmdStanModel(stan_file="ultra_model_fix_mean.stan")

print("Starting HMC/NUTS Sampler...")
fit = model.sample(
    data=stan_data,
    chains=2,
    iter_warmup=300,
    iter_sampling=300,
    max_treedepth=10,
    adapt_delta=0.85,
    show_progress=True
)

# ==========================================
# 6. Check Parameter Recovery
# ==========================================
print("\n=== Sampling Complete ===")
print(fit.diagnose())

# Extract posterior summary
summ = fit.summary()

print("\n=== Parameter Recovery Comparison ===")

# Combine True vs Estimated into a clean DataFrame for Structural Parameters
recovery_data = {
    'Parameter': [
        'Gamma[1,1] (Drift Base 1)', 
        'Gamma[2,2] (Drift Base 2)', 
        'Gamma[2,1] (Oscillation)',
        'Sigma[1,1] (Diffusion 1)',
        'Sigma[2,2] (Diffusion 2)'
    ],
    'True Value': [
        Gamma_true[0,0], 
        Gamma_true[1,1], 
        Gamma_true[1,0],
        Sigma_true[0,0],
        Sigma_true[1,1]
    ],
    'Estimated Mean': [
        summ.loc['Gamma[1,1]', 'Mean'],
        summ.loc['Gamma[2,2]', 'Mean'],
        summ.loc['Gamma[2,1]', 'Mean'],
        summ.loc['Sigma[1,1]', 'Mean'],
        summ.loc['Sigma[2,2]', 'Mean']
    ],
    '95% CI Lower': [
        summ.loc['Gamma[1,1]', '5%'],
        summ.loc['Gamma[2,2]', '5%'],
        summ.loc['Gamma[2,1]', '5%'],
        summ.loc['Sigma[1,1]', '5%'],
        summ.loc['Sigma[2,2]', '5%']
    ],
    '95% CI Upper': [
        summ.loc['Gamma[1,1]', '95%'],
        summ.loc['Gamma[2,2]', '95%'],
        summ.loc['Gamma[2,1]', '95%'],
        summ.loc['Sigma[1,1]', '95%'],
        summ.loc['Sigma[2,2]', '95%']
    ]
}

df_structural = pd.DataFrame(recovery_data)
print("\n-- Structural OU Parameters --")
print(df_structural.to_string(index=False))

# Check Loadings (Lambda)
lambda_est = [summ.loc[f'lambda[{k+1}]', 'Mean'] for k in range(K)]
df_lambda = pd.DataFrame({
    'Item': [f'Item {k+1}' for k in range(K)],
    'True Lambda': lambda_true,
    'Estimated Lambda': lambda_est
})

print("\n-- Factor Loadings (Lambda) --")
print(df_lambda.to_string(index=False))

19:03:18 - cmdstanpy - INFO - compiling stan file /nfs/nfs9/home/nobackup/zwu1/gnpc_explor/src/ultra_model_fix_mean.stan to exe file /nfs/nfs9/home/nobackup/zwu1/gnpc_explor/src/ultra_model_fix_mean


Generating True Continuous-Time LOU Data...
Compiling ultra_model_fix_mean.stan...


19:04:30 - cmdstanpy - INFO - compiled model executable: /nfs/nfs9/home/nobackup/zwu1/gnpc_explor/src/ultra_model_fix_mean
19:04:30 - cmdstanpy - WARNING - Stan compiler has produced 1 warnings:
19:04:30 - cmdstanpy - WARNING - 
--- Translating Stan model to C++ code ---
bin/stanc --filename-in-msg=ultra_model_fix_mean.stan --o=/nfs/nfs9/home/nobackup/zwu1/gnpc_explor/src/ultra_model_fix_mean.hpp /nfs/nfs9/home/nobackup/zwu1/gnpc_explor/src/ultra_model_fix_mean.stan
Warning in 'ultra_model_fix_mean.stan', line 91, column 11 to column 22:
    Found int division:
        R * (R - 1) / 2
    Values will be rounded towards zero. If rounding is not desired you can
    write the division as
        R * (R - 1) / 2.0
    If rounding is intended please use the integer division operator %/%.

--- Compiling C++ code ---
/usr/bin/g++ -std=c++17 -pthread -D_REENTRANT -Wno-sign-compare -Wno-ignored-attributes -Wno-class-memaccess      -I stan/lib/stan_math/lib/tbb_2020.3/include    -O3 -I src -I st

Starting HMC/NUTS Sampler...


19:04:30 - cmdstanpy - INFO - CmdStan start processing
chain 2: 100%|██████████| 600/600 [06:37<00:00,  1.51it/s, (Sampling completed)]


19:11:08 - cmdstanpy - INFO - CmdStan done processing.
19:11:08 - cmdstanpy - WARNING - Non-fatal error during sampling:
Exception: cholesky_decompose: A is not symmetric. A[1,2] = -nan, but A[2,1] = -nan (in 'ultra_model_fix_mean.stan', line 163, column 8 to column 57)
	Exception: cholesky_decompose: A is not symmetric. A[1,2] = -nan, but A[2,1] = -nan (in 'ultra_model_fix_mean.stan', line 163, column 8 to column 57)
	Exception: cholesky_decompose: A is not symmetric. A[1,2] = -nan, but A[2,1] = -nan (in 'ultra_model_fix_mean.stan', line 163, column 8 to column 57)
	Exception: lkj_corr_cholesky_lpdf: Random variable[2] is 0, but must be positive! (in 'ultra_model_fix_mean.stan', line 209, column 4 to column 38)
	Exception: cholesky_decompose: Matrix m is not positive definite (in 'ultra_model_fix_mean.stan', line 179, column 16 to column 77)
	Exception: cholesky_decompose: A is not symmetric. A[1,2] = -inf, but A[2,1] = -inf (in 'ultra_model_fix_mean.stan', line 179, column 16 to col

19:11:08 - cmdstanpy - WARNING - Some chains may have failed to converge.
	Chain 2 had 12 divergent transitions (4.0%)
	Use the "diagnose()" method on the CmdStanMCMC object to see further information.



=== Sampling Complete ===
Checking sampler transitions treedepth.
Treedepth satisfactory for all transitions.

Checking sampler transitions for divergences.
12 of 600 (2.00%) transitions ended with a divergence.
These divergent transitions indicate that HMC is not fully able to explore the posterior distribution.
Try increasing adapt delta closer to 1.
If this doesn't remove all divergences, try to reparameterize the model.

Checking E-BFMI - sampler transitions HMC potential energy.
E-BFMI satisfactory.

Rank-normalized split effective sample size satisfactory for all parameters.

The following parameters had rank-normalized split R-hat greater than 1.01:
  theta2, theta4[1], lambda_free[4], lambda_free[5], sigma_lambda, beta[1,1], beta[3,1], beta[1,2], beta[7,2], b_raw[7,1], b_raw[9,1], b_raw[13,1], b_raw[17,1], b_raw[19,1], b_raw[20,1], b_raw[21,1], b_raw[24,1], b_raw[33,1], b_raw[36,1], b_raw[37,1], b_raw[45,1], b_raw[47,1], b_raw[1,2], b_raw[12,2], b_raw[14,2], b_raw[16,2], b_raw